In [1]:
import numpy as np
import cv2
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor


In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [3]:
class Neural(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(
      nn.Linear(784,128),
      nn.ReLU(),
      nn.Linear(128,128),
      nn.ReLU(),
      nn.Linear(128,10)
    )

  def forward(self,x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits


In [4]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.train()
    total_loss = 0
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)
        total_loss +=loss
        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        # if batch % 100 == 0:
        #     loss, current = loss.item(), (batch + 1) * len(X)
    print(f"avg_loss: {total_loss/num_batches:>7f}\n")

In [5]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [6]:
model = Neural().to(device)
print(model)

Neural(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [7]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [8]:
def true_vals(array):
  array = array.flatten()
  true_values = np.zeros((array.size,array.max()+1))
  true_values[np.arange(array.size),array] = 1
  return true_values

## TRAINING ON MNIST DATASET

In [20]:
# Download training data from open datasets.
training_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# Download test data from open datasets.
test_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

In [21]:
batch_size = 128

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([128, 1, 28, 28])
Shape of y: torch.Size([128]) torch.int64


In [23]:
print(len(train_dataloader))

469


In [11]:
epochs = 100
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
print("Done!")

Epoch 1
-------------------------------
avg_loss: 2.296880

Epoch 2
-------------------------------
avg_loss: 2.283559

Epoch 3
-------------------------------
avg_loss: 2.268229

Epoch 4
-------------------------------
avg_loss: 2.249177

Epoch 5
-------------------------------
avg_loss: 2.224624

Epoch 6
-------------------------------
avg_loss: 2.191909

Epoch 7
-------------------------------
avg_loss: 2.147456

Epoch 8
-------------------------------
avg_loss: 2.087183

Epoch 9
-------------------------------
avg_loss: 2.007633

Epoch 10
-------------------------------
avg_loss: 1.906971

Epoch 11
-------------------------------
avg_loss: 1.786688

Epoch 12
-------------------------------
avg_loss: 1.653390

Epoch 13
-------------------------------
avg_loss: 1.516272

Epoch 14
-------------------------------
avg_loss: 1.383399

Epoch 15
-------------------------------
avg_loss: 1.259876

Epoch 16
-------------------------------
avg_loss: 1.148425

Epoch 17
------------------------

In [12]:
test(test_dataloader,model,loss_fn)

Test Error: 
 Accuracy: 91.6%, Avg loss: 0.288314 



In [13]:
torch.save(model.state_dict(), "model_mnist.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


## TRAINING ON EMNIST DATASET

In [16]:
# Download training data from open datasets.
training_data = datasets.EMNIST(
    root="data",
    split='digits',
    train=True,
    download=True,
    transform=ToTensor(),
)

# Download test data from open datasets.
test_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

In [26]:
batch_size = 256

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([256, 1, 28, 28])
Shape of y: torch.Size([256]) torch.int64


In [27]:
print(len(train_dataloader))

235


In [28]:
epochs = 100
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    # test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
avg_loss: 1.117804

Epoch 2
-------------------------------
avg_loss: 0.803379

Epoch 3
-------------------------------
avg_loss: 0.682734

Epoch 4
-------------------------------
avg_loss: 0.606725

Epoch 5
-------------------------------
avg_loss: 0.554395

Epoch 6
-------------------------------
avg_loss: 0.516097

Epoch 7
-------------------------------
avg_loss: 0.486822

Epoch 8
-------------------------------
avg_loss: 0.463703

Epoch 9
-------------------------------
avg_loss: 0.444964

Epoch 10
-------------------------------
avg_loss: 0.429504

Epoch 11
-------------------------------
avg_loss: 0.416521

Epoch 12
-------------------------------
avg_loss: 0.405450

Epoch 13
-------------------------------
avg_loss: 0.395901

Epoch 14
-------------------------------
avg_loss: 0.387584

Epoch 15
-------------------------------
avg_loss: 0.380277

Epoch 16
-------------------------------
avg_loss: 0.373796

Epoch 17
------------------------

In [29]:
test(test_dataloader, model, loss_fn)


Test Error: 
 Accuracy: 92.6%, Avg loss: 0.250690 



In [30]:
torch.save(model.state_dict(), "model_emnist.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth
